# HaluRISC — Full Training Pipeline (Colab) — corrected grouped-split protocol

Runs the complete experiment protocol on a Colab GPU (T4 or better):

1. HaluEval download + prepare with a **GROUP-AWARE 70/15/15 split** (both answers of one question stay in the same partition; a leakage report is generated and asserted)
2. Full feature extraction (length, lexical, entity/NER, **NLI**, numeric, hedging, semantic) + NLI checkpoint provenance
3. XGBoost tuning (30 iters, 5-fold CV) + baselines + 3-seed protocol + early stopping
4. Platt vs isotonic calibration, ECE/Brier, McNemar, bootstrap CIs, Wilcoxon, 7-group ablations
5. SHAP global + local explanations
6. RAGTruth zero-shot external validation
7. Error analysis (10 FP + 10 FN, auto-tagged for manual review)
8. Latency/efficiency analysis
9. Optional LLM-as-judge comparison (needs OPENAI_API_KEY)
10. Artifact manifest generation (hashes, versions, hardware, split report)
11. Zip artifacts + feature matrix to **Google Drive** for download

**Before starting:** have `colab/halurisc_src.zip` from the repo ready. Cell 3 opens a file-picker to select it from your laptop (no Drive upload needed). Alternatively, upload it once to `MyDrive/HaluRISC/halurisc_src.zip` and it will be picked up automatically.

**Runtime:** enable GPU (Runtime > Change runtime type). **Use T4 (16 GB) or L4 (24 GB) if available — an A100 is overkill for this workload** (the models are small by design; total VRAM use is only ~2 GB). Total wall time ≈ 15-30 min.

**After the run:** download the zip, unzip at the repo root of your laptop, then start the API: `& .venv\Scripts\python.exe -m uvicorn src.api.main:app --port 8000`.


In [ ]:
# 1) Mount Google Drive (artifacts persist here across sessions)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/HaluRISC'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive mounted at', DRIVE_DIR)


In [ ]:
# 2) Environment check: GPU must be enabled
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if not torch.cuda.is_available():
    print('!! No GPU detected - enable GPU in Runtime > Change runtime type')
    raise SystemExit(1)


In [ ]:
# 3) Get the HaluRISC source. PREFERRED: browser file-picker (select colab/halurisc_src.zip from your laptop).
#    Fallback: if you already put halurisc_src.zip inside Drive/HaluRISC/, it is used automatically.
import zipfile, io, os
from google.colab import files

ROOT = '/content/HaluRISC'
os.makedirs(ROOT, exist_ok=True)

if not os.path.exists(os.path.join(ROOT, 'src')):
    src_zip = None
    drive_zip = os.path.join(DRIVE_DIR, 'halurisc_src.zip')
    if os.path.exists(drive_zip):
        src_zip = open(drive_zip, 'rb')
        print('Found halurisc_src.zip on Drive - extracting...')
        with zipfile.ZipFile(src_zip) as z:
            z.extractall(ROOT)
    else:
        print('Upload halurisc_src.zip (repo/colab/halurisc_src.zip) - use the file picker:')
        uploaded = files.upload()
        for name, content in uploaded.items():
            with zipfile.ZipFile(io.BytesIO(content)) as z:
                z.extractall(ROOT)
    print('Extracted to', ROOT)
print('src present:', os.path.exists(os.path.join(ROOT, 'src')))
%cd {ROOT}


In [ ]:
# 4) Install pinned dependencies (Colab keeps its own torch)
!pip install -q -r colab/requirements-colab.txt
!python -m spacy download en_core_web_sm -q
print('deps OK')


In [ ]:
# 5) HaluEval download + prepare with GROUP-AWARE split (item_idx) + integrity check
!python src/data/download.py
!python src/data/prepare.py
import json
rep = json.load(open('artifacts/split_integrity_report.json'))
print(json.dumps(rep, indent=2))
assert rep['leakage_free'] and rep['groups_spanning_multiple_splits'] == 0, 'Split leakage detected!'


In [ ]:
# 5b) Restore cached HaluEval features from Drive when valid (cell 6 shortcut)
# Saves 5-10 min on repeat runs. The cache is VERIFIED against the freshly built
# qa_clean.parquet (rows / sample_ids / labels / splits / leakage) - a mismatch
# falls back to full extraction in cell 6. Set CACHE_OK = True on success.
import os, sys, json, shutil
sys.path.insert(0, '.')
from colab.drive_cache import verify_halueval_features

DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
os.makedirs(DRIVE_CACHE, exist_ok=True)
cache_feat = os.path.join(DRIVE_CACHE, 'features_full.parquet')
CACHE_OK = False
if os.path.exists(cache_feat):
    tmp = '/content/_cache_features.parquet'
    shutil.copy(cache_feat, tmp)
    v = verify_halueval_features(tmp, 'data/processed/qa_clean.parquet')
    if v['ok']:
        shutil.move(tmp, 'data/processed/features_full.parquet')
        cache_nli = os.path.join(DRIVE_CACHE, 'nli_model_used.json')
        if os.path.exists(cache_nli):
            shutil.copy(cache_nli, 'data/processed/nli_model_used.json')
        CACHE_OK = True
        print('FEATURE CACHE RESTORED from Drive and verified against qa_clean.')
    else:
        print('Cached features INVALID - cell 6 will re-extract:', v['checks'])
        os.remove(tmp)
else:
    print('No HaluEval feature cache on Drive yet - cell 6 will extract.')


In [ ]:
# 6) Full feature extraction (7 groups, ~40K NLI pairs on GPU; ~5-10 min on T4/L4)
# Skipped automatically when cell 5b restored a verified Drive cache.
# --device cuda -> models load in fp16 + CUDA; --batch-size 256 -> bigger GPU batches = faster
# Default NLI model: cross-encoder/nli-deberta-v3-base. Fallback: --nli-model cross-encoder/nli-MiniLM2-L6-H768
import json
if CACHE_OK:
    print('Using Drive-cached features (verified against qa_clean.parquet).')
    print('NLI provenance:', json.load(open('data/processed/nli_model_used.json')))
else:
    get_ipython().system('python src/features/extract_features.py --device cuda --batch-size 256')
    print('NLI provenance:', json.load(open('data/processed/nli_model_used.json')))


In [ ]:
# 6b) Upload the HaluEval feature cache to Drive (next session skips extraction)
# Safe: features_full.parquet contains no FaithBench text (HaluEval is MIT).
import os, shutil
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
os.makedirs(DRIVE_CACHE, exist_ok=True)
shutil.copy('data/processed/features_full.parquet', os.path.join(DRIVE_CACHE, 'features_full.parquet'))
shutil.copy('data/processed/nli_model_used.json', os.path.join(DRIVE_CACHE, 'nli_model_used.json'))
print('Uploaded HaluEval feature cache to Drive (halurisc_cache/).')


In [ ]:
# 7.0) Restore Version A cell-7 artifacts from Drive when valid
# Cell 7 is optional for B2-B4, but if you run the full 1-15 workflow this
# checkpoint prevents repeating its CPU training after a runtime crash.
import json, os, shutil
from colab.drive_cache import version_a_restore_valid, sha256_file

DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
VA_DIR = os.path.join(DRIVE_CACHE, 'version_a')
VA_MARKER = os.path.join(VA_DIR, 'marker.json')
VA_FILES = [
    ('models/model_xgboost_raw.joblib', 'artifacts/models/model_xgboost_raw.joblib'),
    ('models/model_xgboost_calibrated.joblib', 'artifacts/models/model_xgboost_calibrated.joblib'),
    ('models/calibrator_platt.joblib', 'artifacts/models/calibrator_platt.joblib'),
    ('models/scaler.joblib', 'artifacts/models/scaler.joblib'),
    ('models/params.json', 'artifacts/models/params.json'),
    ('models/feature_names.json', 'artifacts/models/feature_names.json'),
    ('results/final_results.json', 'artifacts/results/final_results.json'),
    ('results/seed_metrics.json', 'artifacts/results/seed_metrics.json'),
    ('results/model_comparison.csv', 'artifacts/results/model_comparison.csv'),
    ('results/ablation_results.csv', 'artifacts/results/ablation_results.csv'),
]
VA_OK = False
if os.path.exists(VA_MARKER) and version_a_restore_valid(
    VA_MARKER, 'data/processed/features_full.parquet', 'data/processed/qa_clean.parquet',
    'artifacts/split_integrity_report.json', VA_DIR
):
    for rel, local in VA_FILES:
        os.makedirs(os.path.dirname(local), exist_ok=True)
        shutil.copy(os.path.join(VA_DIR, rel), local)
    VA_OK = True
    print('Version A cell-7 artifacts RESTORED from Drive (input hashes verified).')
else:
    print('No valid Version A cache on Drive - cell 7 will run if selected.')


In [ ]:
# 7) Full Version A experiment (optional; CPU, ~10-20 min)
# HALU_XGB_DEVICE=cpu keeps boosters portable after download.
# RESUMABLE: restores from 7.0 or checkpoints immediately after success.
import datetime, json, os, shutil
os.environ['HALU_XGB_DEVICE'] = 'cpu'
if VA_OK:
    print('Version A cell 7 already complete - skipping training.')
else:
    rc = os.system('python src/models/train_pipeline.py')
    if rc != 0:
        raise RuntimeError(f'Version A cell 7 failed with exit code {rc}')
    os.makedirs(VA_DIR, exist_ok=True)
    for rel, local in VA_FILES:
        dest = os.path.join(VA_DIR, rel)
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        shutil.copy(local, dest)
    marker = {
        'schema': 'version-a-checkpoint-v1',
        'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
        'inputs': {
            'features_full.parquet': sha256_file('data/processed/features_full.parquet'),
            'qa_clean.parquet': sha256_file('data/processed/qa_clean.parquet'),
            'split_integrity_report.json': sha256_file('artifacts/split_integrity_report.json'),
        },
        'artifacts': [rel for rel, _ in VA_FILES],
    }
    with open(VA_MARKER, 'w', encoding='utf-8') as f:
        json.dump(marker, f, indent=2)
    print('Checkpointed Version A cell-7 artifacts to Drive (halurisc_cache/version_a).')


In [ ]:
# 7b.0) Restore B2 artifacts from Drive when valid (7b shortcut)
# B2 is reusable only when its cached run consumed THIS features_full.parquet
# (hash check). Restores results + models; sets B2_OK=True to skip training.
import os, json, shutil
from colab.drive_cache import b2_restore_valid
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
b2_dir = os.path.join(DRIVE_CACHE, 'b2')
B2_OK = False
cfg = os.path.join(b2_dir, 'b2_run_config.json')
if (os.path.exists(cfg) and os.path.isdir(os.path.join(DRIVE_CACHE, 'b2_models'))
    and b2_restore_valid(cfg, 'data/processed/features_full.parquet')):
    shutil.copytree(b2_dir, 'artifacts/results/b2', dirs_exist_ok=True)
    shutil.copytree(os.path.join(DRIVE_CACHE, 'b2_models'), 'artifacts/models/b2', dirs_exist_ok=True)
    B2_OK = True
    print('B2 artifacts RESTORED from Drive (feature hash verified).')
else:
    print('No valid B2 cache on Drive yet - cell 7b will train (8-15 min).')


In [ ]:
# 7b) B2: corrected baselines + artifact controls (grouped CV, TF-IDF shortcut checks)
# Runs: majority, overlap heuristic, TF-IDF (all/answer/context), NLI-only, LR, RF, tuned XGBoost
# seeds 42/123/456. Grouped 5-fold CV keyed by item_idx; thresholds: 0.5 / overlap tuned on val.
# ~8-15 min on T4/L4. All outputs under artifacts/{results,models}/b2 (Version A untouched).
# RESUMABLE: skipped automatically when cell 7b.0 restored valid artifacts (B2_OK).
import os
os.environ['HALU_XGB_DEVICE'] = 'cpu'  # portable boosters, see cell 7 note
if B2_OK:
    print('B2 already complete (restored from Drive) - skipping training.')
else:
    get_ipython().system('python src/models/run_b2_baselines.py')


In [ ]:
# 7b.5) Upload B2 artifacts to Drive (crash-safe checkpoint)
import os, shutil
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
os.makedirs(os.path.join(DRIVE_CACHE, 'b2'), exist_ok=True)
shutil.copytree('artifacts/results/b2', os.path.join(DRIVE_CACHE, 'b2'), dirs_exist_ok=True)
shutil.copytree('artifacts/models/b2', os.path.join(DRIVE_CACHE, 'b2_models'), dirs_exist_ok=True)
print('Checkpointed B2 artifacts to Drive (halurisc_cache/b2).')


In [ ]:
# 7c) Show B2 headline results (comparison table + leakage-removal impact)
import json, pandas as pd
comp = pd.read_csv('artifacts/results/b2/b2_model_comparison.csv', index_col=0)
cols = [c for c in ['precision_mean','recall_mean','f1_mean','auroc_mean','pr_auc_mean','mcc_mean','ece_mean','threshold'] if c in comp.columns]
print('B2 MODEL COMPARISON (test set, seeds 42/123/456)')
print(comp[cols].round(4).to_string())
print('\nLEAKAGE-REMOVAL IMPACT:')
print(json.dumps(json.load(open('artifacts/results/b2/b2_leakage_comparison.json')), indent=2))


In [ ]:
# 7d) B1: build the unified external dataset (official RAGTruth + FaithBench)
# Required by B3. Downloads RAGTruth response/source files (~35 MB) and FaithBench
# annotation batches (~3 MB). Raw files stay in the Colab VM (never committed;
# FaithBench is CC BY-NC-SA). Deterministic: rerunning gives byte-identical output.
!python src/data/download_ragtruth.py
!python src/data/download_faithbench.py
!python src/data/prepare_unified.py


In [ ]:
# 7d.5) Restore cached B3 external features from Drive when valid (7e shortcut)
# B3 extraction is the heaviest step (~15-40 min). The cache stores metadata +
# features ONLY (no FaithBench text, CC BY-NC-SA) and is reused only when its
# unified-parquet hash matches the freshly built unified_records.parquet.
import os, json, shutil, hashlib
from colab.drive_cache import b3_feature_cache_safe
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
cache_meta = os.path.join(DRIVE_CACHE, 'b3_external_features.meta.json')
cache_features = os.path.join(DRIVE_CACHE, 'b3_external_features.parquet')
B3_CACHE_OK = False
if (os.path.exists(cache_meta) and os.path.exists(cache_features)
    and b3_feature_cache_safe(cache_features)):
    meta = json.load(open(cache_meta))
    unified_sha = hashlib.sha256(open('data/processed/unified_records.parquet','rb').read()).hexdigest()
    if meta.get('input_sha256') == unified_sha:
        shutil.copy(cache_features, 'data/processed/b3_external_features.parquet')
        shutil.copy(cache_meta, 'data/processed/b3_external_features.meta.json')
        B3_CACHE_OK = True
        print('B3 external feature cache restored and hash-verified.')
    else:
        print('B3 cache stale (unified parquet changed) - cell 7e will re-extract.')
else:
    print('No B3 external feature cache on Drive yet - cell 7e will extract.')


In [ ]:
# 7d.6) Restore B3 RESULTS from Drive when valid (7e shortcut)
# B3 results are reusable only when the unified parquet AND the B2 model hashes
# match the cached run. Restores results + figures; sets B3_OK=True to skip 7e.
import os, json, shutil
from colab.drive_cache import b3_restore_valid, b3_results_safe
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
b3_dir = os.path.join(DRIVE_CACHE, 'b3')
figures_dir = os.path.join(DRIVE_CACHE, 'b3_figures')
B3_OK = False
cfg = os.path.join(b3_dir, 'b3_run_config.json')
if (os.path.exists(cfg) and os.path.exists(os.path.join(b3_dir, 'b3_predictions.parquet'))
    and os.path.isdir(figures_dir) and b3_results_safe(b3_dir)
    and b3_restore_valid(cfg, 'data/processed/unified_records.parquet', 'artifacts/models/b2')):
    shutil.copytree(b3_dir, 'artifacts/results/b3', dirs_exist_ok=True)
    shutil.copytree(os.path.join(DRIVE_CACHE, 'b3_figures'), 'artifacts/figures/b3', dirs_exist_ok=True)
    B3_OK = True
    print('B3 results RESTORED from Drive (unified + B2 model hashes verified).')
else:
    print('No valid B3 results cache on Drive yet - cell 7e will run.')


In [ ]:
# 7e) B3: cross-domain zero-shot evaluation (HEAVY: ~18.5K external rows)
# Extracts the 26 features on RAGTruth + FaithBench (T4/L4: ~15-40 min), then
# evaluates the B2 XGBoost models zero-shot (fixed threshold 0.5, source-group
# bootstrap CIs, subgroup + transfer-failure analysis, FaithBench label sensitivity).
# REQUIRES cell 7b to have run first in this session: B2 must train with
# HALU_XGB_DEVICE=cpu so the saved boosters load here (CUDA boosters do not port).
# RESUMABLE: skipped when cell 7d.6 restored valid results (B3_OK); uses
# --skip-features when cell 7d.5 restored the feature cache (B3_CACHE_OK).
import os, shutil
os.environ['HALU_XGB_DEVICE'] = 'cpu'
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
os.makedirs(DRIVE_CACHE, exist_ok=True)
if B3_OK:
    print('B3 already complete (results restored from Drive) - skipping run.')
else:
    flag = '--skip-features' if B3_CACHE_OK else ''
    cmd = f'python src/models/run_b3_cross_domain.py --device cuda --batch-size 256 {flag}'.strip()
    print('Running:', cmd)
    rc = os.system(cmd)
    if rc != 0:
        cache_src = 'data/processed/b3_external_features.parquet'
        if os.path.exists(cache_src):
            os.makedirs(DRIVE_CACHE, exist_ok=True)
            shutil.copy(cache_src, os.path.join(DRIVE_CACHE, 'b3_external_features.parquet'))
            shutil.copy('data/processed/b3_external_features.meta.json', os.path.join(DRIVE_CACHE, 'b3_external_features.meta.json'))
            print('B3 failed after feature cache creation; feature cache checkpointed.')
        raise RuntimeError(f'B3 run failed with exit code {rc}')
    shutil.copytree('artifacts/results/b3', os.path.join(DRIVE_CACHE, 'b3'), dirs_exist_ok=True)
    shutil.copytree('artifacts/figures/b3', os.path.join(DRIVE_CACHE, 'b3_figures'), dirs_exist_ok=True)
    print('Checkpointed B3 results + figures to Drive (halurisc_cache/b3).')
if os.path.exists('data/processed/b3_external_features.parquet'):
    shutil.copy('data/processed/b3_external_features.parquet', os.path.join(DRIVE_CACHE, 'b3_external_features.parquet'))
    shutil.copy('data/processed/b3_external_features.meta.json', os.path.join(DRIVE_CACHE, 'b3_external_features.meta.json'))
    print('Uploaded B3 external feature cache to Drive.')


In [ ]:
# 7f) Show B3 headline results (dataset metrics + transfer comparison)
import json, pandas as pd
m = json.load(open('artifacts/results/b3/b3_dataset_metrics.json'))
keep = ('n_rows','n_groups','f1_mean','auroc_mean','ece_mean','predicted_positive_rate','label_positive_rate')
rows = {k: {kk: (round(vv,4) if isinstance(vv,float) else vv) for kk,vv in v.items() if kk in keep} for k,v in m.items() if v}
print('B3 DATASET METRICS (zero-shot, B2 XGBoost, seeds 42/123/456)')
print(pd.DataFrame(rows).T.to_string())
print('\nTRANSFER COMPARISON:')
print(pd.read_csv('artifacts/results/b3/b3_transfer_comparison.csv').to_string(index=False))
print('\nFAITHBENCH LABEL SENSITIVITY:')
print(json.dumps(json.load(open('artifacts/results/b3/b3_label_sensitivity.json')), indent=2))


In [ ]:
# 7g.0) Restore B4 artifacts from Drive when valid (7g shortcut)
# B4 is reusable only when produced from THIS b3_predictions.parquet (hash check).
import os, json, shutil
from colab.drive_cache import b4_restore_valid
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
b4_dir = os.path.join(DRIVE_CACHE, 'b4')
B4_OK = False
cfg = os.path.join(b4_dir, 'b4_run_config.json')
if os.path.exists(cfg) and b4_restore_valid(cfg, 'artifacts/results/b3/b3_predictions.parquet'):
    shutil.copytree(b4_dir, 'artifacts/results/b4', dirs_exist_ok=True)
    shutil.copytree(os.path.join(DRIVE_CACHE, 'b4_models'), 'artifacts/models/b4', dirs_exist_ok=True)
    shutil.copytree(os.path.join(DRIVE_CACHE, 'b4_figures'), 'artifacts/figures/b4', dirs_exist_ok=True)
    B4_OK = True
    print('B4 artifacts RESTORED from Drive (b3 predictions hash verified).')
else:
    print('No valid B4 cache on Drive yet - cell 7g will run.')


In [ ]:
# 7g) B4: calibration under distribution shift
# Reuses B3 predictions, cached external features, and B2 models (NO new
# feature extraction, NO retraining). Fits calibrators ONLY on HaluEval
# validation (source) and RAGTruth QA train (target, disjoint source
# groups), then evaluates on HaluEval test / RAGTruth QA test / FaithBench.
# Fast on CPU (2-5 min). All B4 calibrators are pure sklearn -> portable.
# RESUMABLE: skipped when cell 7g.0 restored valid artifacts (B4_OK);
# on crash a b4_crash.log is written with the full traceback.
import os, shutil
os.environ['HALU_XGB_DEVICE'] = 'cpu'
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'halurisc_cache')
os.makedirs(DRIVE_CACHE, exist_ok=True)
if B4_OK:
    print('B4 already complete (artifacts restored from Drive) - skipping run.')
else:
    rc = os.system('python src/models/run_b4_calibration_shift.py')
    if rc != 0:
        crash = 'artifacts/results/b4/b4_crash.log'
        if os.path.exists(crash):
            shutil.copy(crash, os.path.join(DRIVE_CACHE, 'b4_crash.log'))
            print('B4 crash log checkpointed to Drive.')
        raise RuntimeError(f'B4 run failed with exit code {rc}')
    shutil.copytree('artifacts/results/b4', os.path.join(DRIVE_CACHE, 'b4'), dirs_exist_ok=True)
    shutil.copytree('artifacts/models/b4', os.path.join(DRIVE_CACHE, 'b4_models'), dirs_exist_ok=True)
    shutil.copytree('artifacts/figures/b4', os.path.join(DRIVE_CACHE, 'b4_figures'), dirs_exist_ok=True)
    print('Checkpointed B4 artifacts to Drive (halurisc_cache/b4).')


In [ ]:
# 7h) Show B4 headline results (calibration metrics + target calibration)
import json, pandas as pd
m = json.load(open('artifacts/results/b4/b4_calibration_metrics.json'))
rows = []
for subset, methods in m.items():
    for method, mm in methods.items():
        rows.append({'subset': subset, 'method': method,
                     'ece': mm.get('ece_mean'), 'ace': mm.get('ace_mean'),
                     'brier': mm.get('brier_mean'), 'nll': mm.get('nll_mean'),
                     'slope': mm.get('slope_mean'), 'f1': mm.get('f1_mean')})
df = pd.DataFrame(rows).dropna(subset=['ece']).round(4)
print('B4 CALIBRATION METRICS (seeds 42/123/456; calibrators on HaluEval val only)')
print(df.to_string(index=False))
print('\nTARGET CALIBRATION (RAGTruth QA train -> test):')
t = json.load(open('artifacts/results/b4/b4_target_calibration.json'))
print('n_cal=%d n_test=%d overlap_groups_removed=%d' % (t['n_calibration_rows'], t['n_test_rows'], t['overlapping_groups_removed']))
for k, v in t['methods'].items():
    if 'ece_mean' in v:
        print('  %s: ece=%.4f brier=%.4f' % (k, v['ece_mean'], v['brier_mean']))


In [ ]:
# 7i) Verify every artifact loads on THIS machine BEFORE packaging
# Loads the B2 boosters, B4 calibrators, and all prediction parquet files;
# proves they predict correctly (prevents the old post-download loading error).
# Exit code 0 = safe to package and use locally after download.
!python src/models/verify_artifacts.py


### CRASH RECOVERY (resume from Drive)

If the runtime dies mid-run, restart it and run cells **1-5**, **5b**, **6/6b**, then **7.0** and **7** if you want Version A. Completed phases restore automatically:

- **7.0** restores Version A cell-7 artifacts; cell **7** skips training.
- **7b.0** restores B2; cell **7b** skips training.
- **7d.5** restores B3 features; **7d.6** restores B3 results; cell **7e** skips extraction/evaluation.
- **7g.0** restores B4; cell **7g** skips calibration.

Continue from the first cell whose checkpoint is missing. Every completed heavy phase is written to `Drive/halurisc_cache/` immediately.


In [ ]:
# 8) SHAP explanations + calibration/ROC/PR figures
!python src/explain/shap_analysis.py


In [ ]:
# 9) RAGTruth zero-shot external validation (downloads from HuggingFace)
!python src/data/download_ragtruth.py
!python src/models/eval_ragtruth.py


In [ ]:
# 10) Error analysis (10 FP + 10 FN, auto-tagged with the blueprint taxonomy)
# NOTE: categories are heuristic and must be manually reviewed in
# artifacts/results/error_analysis_cases.json before the paper uses them.
!python src/models/error_analysis.py


In [ ]:
# 11) Latency / efficiency analysis (features + predict + SHAP, p50/p95)
!python src/models/eval_efficiency.py


In [ ]:
# 12) OPTIONAL: LLM-as-judge comparison (200 samples, ~$0.05-0.15)
# Requires OPENAI_API_KEY. In Colab: Edit > Notebook settings > Secrets, add OPENAI_API_KEY.
import os
if os.environ.get('OPENAI_API_KEY'):
    os.system('python src/models/eval_llm_judge.py')
else:
    print('OPENAI_API_KEY not set - skipping optional LLM-as-judge. You can run it later locally.')


In [ ]:
# 13) Generate the artifact manifest (hashes, versions, hardware, split report)
!python src/models/make_manifest.py
import json
m = json.load(open('artifacts/results/manifest.json'))
print(json.dumps({k: m[k] for k in ['generated_at', 'git_commit', 'model_version', 'nli_model', 'split_report']}, indent=2))


In [ ]:
# 14) Show the headline results (model comparison, calibration, RAGTruth, per-seed rows)
import json, pandas as pd
res = json.load(open('artifacts/results/final_results.json'))
rows = {k: v for k, v in res.items() if isinstance(v, dict) and 'f1' in v and 'auroc' in v}
df = pd.DataFrame(rows).T[['precision','recall','f1','auroc','pr_auc','mcc']].round(4)
print('MODEL COMPARISON (test set, mean over seeds 42/123/456)')
print(df.to_string())
print('\nCALIBRATION:')
print(json.dumps(res['calibration'], indent=2))
print('\nABLATION:')
print(pd.DataFrame(res['ablation']).to_string(index=False))
print('\nPER-SEED XGBOOST:')
print(pd.DataFrame(json.load(open('artifacts/results/seed_metrics.json'))['xgboost']).round(4).to_string(index=False))
rt = json.load(open('artifacts/results/ragtruth_results.json'))
print('\nRAGTRUTH ZERO-SHOT: f1=%.4f auroc=%.4f ece=%.4f (n=%d)' % (rt['f1'], rt['auroc'], rt['ece'], rt['n_samples']))


In [ ]:
# 15) Package artifacts to Drive (persists across sessions) and offer a download link
import os, zipfile
from datetime import date
from google.colab import files

stamp = date.today().isoformat()
zip_path = f'{DRIVE_DIR}/halurisc_artifacts_{stamp}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, fnames in os.walk('artifacts'):
        for fn in fnames:
            p = os.path.join(root, fn)
            z.write(p, os.path.relpath(p, '.'))
    for fn in ['data/processed/features_full.parquet', 'data/processed/qa_clean.parquet',
               'data/processed/nli_model_used.json', 'data/processed/audit_50_samples.json']:
        if os.path.exists(fn):
            z.write(fn)
print('Saved:', zip_path, f'({os.path.getsize(zip_path)/1e6:.1f} MB)')
print()
print('NEXT: download the zip from your Drive, unzip at the repo root of your laptop.')
print('The API (uvicorn) and web dashboard will then load the corrected artifacts.')
files.download(zip_path)
